In [1]:
import pandas as pd
import numpy as np

In [2]:
raw = pd.read_excel('Análisis_de_piso.xlsx', header=None)

# Construimos manualmente los encabezados correctos, porque el archivo
# tiene los nombres reales repartidos en distintas filas según el bloque
meses_leads = ['Leads_feb24','Leads_mar24','Leads_abr24','Leads_may24','Leads_jun24',
               'Leads_jul24','Leads_ago24','Leads_sep24','Leads_oct24','Leads_nov24',
               'Leads_dic24','Leads_ene25','Leads_feb25','Leads_mar25','Leads_abr25']

meses_ventas = ['Ventas_feb24','Ventas_mar24','Ventas_abr24','Ventas_may24','Ventas_jun24',
                'Ventas_jul24','Ventas_ago24','Ventas_sep24','Ventas_oct24','Ventas_nov24',
                'Ventas_dic24','Ventas_ene25','Ventas_feb25','Ventas_mar25','Ventas_abr25',
                'Ventas_may25','Ventas_jun25']

columnas_finales = (
    ['Asesor']
    + meses_leads
    + ['LeadsTotal', 'LeadsTotalAjustado', 'RakingLeads', 'RakingLeadsAjustado']
    + meses_ventas
    + ['LeadsPromXMes', 'TVentasPiso', 'Factor', 'TotalVentasHistoricas', 'RakingVentas',
       'DependenciaPiso', 'HistoricoPDM', 'HistoricoSDC', 'ConversionPDM', 'ConversionSDC',
       'FactorPDM', 'FactorSDC', 'RakingPDM', 'RakingSDC']
)

# Los datos de asesores inician en la fila 3 del excel original (después de los encabezados)
df = raw.iloc[3:].copy()
df.columns = columnas_finales
df = df.dropna(subset=['Asesor']).reset_index(drop=True)

df.head()

,Asesor,Leads_feb24,Leads_mar24,Leads_abr24,Leads_may24,Leads_jun24,Leads_jul24,Leads_ago24,Leads_sep24,Leads_oct24,...,RakingVentas,DependenciaPiso,HistoricoPDM,HistoricoSDC,ConversionPDM,ConversionSDC,FactorPDM,FactorSDC,RakingPDM,RakingSDC
0,ABIGAIL,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23,0,1,5,1,0,1.00,5.00,1,20
1,ALAN GONZALES,1,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,...,6,0.61,59,40,0.31,0.02,3.28,2.22,7,10
2,ALICIA LEDESMA,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,1,1,0,0,0.00,0.00,NaN,NaN
3,AURELIO FELIPE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,6,...,11,0.38,28,31,0.29,0.01,3.50,3.88,9,18
4,CARLOS ALBERTO NIETO,NaN,NaN,NaN,2,NaN,NaN,2,8,4,...,8,0.6,64,30,0.23,0.02,4.27,2.00,13,7


In [3]:
# --- Dataset 1: Registro (Leads y Ventas mes a mes) ---
# Un NaN aquí significa "0 actividad ese mes" -> se rellena con 0
columnas_registro = ['Asesor'] + meses_leads + meses_ventas
df_registro = df[columnas_registro].copy()
df_registro = df_registro.fillna(0)

df_registro.isnull().sum()

Asesor          0
Leads_feb24     0
Leads_mar24     0
Leads_abr24     0
Leads_may24     0
Leads_jun24     0
Leads_jul24     0
Leads_ago24     0
Leads_sep24     0
Leads_oct24     0
Leads_nov24     0
Leads_dic24     0
Leads_ene25     0
Leads_feb25     0
Leads_mar25     0
Leads_abr25     0
Ventas_feb24    0
Ventas_mar24    0
Ventas_abr24    0
Ventas_may24    0
Ventas_jun24    0
Ventas_jul24    0
Ventas_ago24    0
Ventas_sep24    0
Ventas_oct24    0
Ventas_nov24    0
Ventas_dic24    0
Ventas_ene25    0
Ventas_feb25    0
Ventas_mar25    0
Ventas_abr25    0
Ventas_may25    0
Ventas_jun25    0
dtype: int64

In [4]:
# --- Dataset 1: Registro (Leads y Ventas mes a mes) ---
columnas_registro = ['Asesor'] + meses_leads + meses_ventas
df_registro = df[columnas_registro].copy()

# Convertimos todas las columnas numéricas forzando el tipo;
# cualquier espacio en blanco, "\xa0" u otro texto raro se vuelve NaN
for col in meses_leads + meses_ventas:
    df_registro[col] = pd.to_numeric(df_registro[col], errors='coerce')

# Ahora sí, todos los NaN reales (incluyendo los que antes eran espacios ocultos)
# se rellenan con 0
df_registro[meses_leads + meses_ventas] = df_registro[meses_leads + meses_ventas].fillna(0)

df_registro.isnull().sum()

Asesor          0
Leads_feb24     0
Leads_mar24     0
Leads_abr24     0
Leads_may24     0
Leads_jun24     0
Leads_jul24     0
Leads_ago24     0
Leads_sep24     0
Leads_oct24     0
Leads_nov24     0
Leads_dic24     0
Leads_ene25     0
Leads_feb25     0
Leads_mar25     0
Leads_abr25     0
Ventas_feb24    0
Ventas_mar24    0
Ventas_abr24    0
Ventas_may24    0
Ventas_jun24    0
Ventas_jul24    0
Ventas_ago24    0
Ventas_sep24    0
Ventas_oct24    0
Ventas_nov24    0
Ventas_dic24    0
Ventas_ene25    0
Ventas_feb25    0
Ventas_mar25    0
Ventas_abr25    0
Ventas_may25    0
Ventas_jun25    0
dtype: int64

In [5]:
# --- Dataset 2: Análisis (Totales, Raking, Conversión, Factor) ---
columnas_analisis = (
    ['Asesor', 'LeadsTotal', 'LeadsTotalAjustado', 'RakingLeads', 'RakingLeadsAjustado']
    + ['LeadsPromXMes', 'TVentasPiso', 'Factor', 'TotalVentasHistoricas', 'RakingVentas',
       'DependenciaPiso', 'HistoricoPDM', 'HistoricoSDC', 'ConversionPDM', 'ConversionSDC',
       'FactorPDM', 'FactorSDC', 'RakingPDM', 'RakingSDC']
)
df_analisis = df[columnas_analisis].copy()

# Columnas de tipo Raking: un NaN significa "no clasificado" -> va al último lugar,
# nunca 0, porque 0 implicaría el mejor lugar. RakingVentas trae '-' en vez de NaN,
# por eso primero se convierte a numérico.
col_raking = ['RakingLeads', 'RakingLeadsAjustado', 'RakingVentas', 'RakingPDM', 'RakingSDC']

for c in col_raking:
    df_analisis[c] = pd.to_numeric(df_analisis[c], errors='coerce')
    df_analisis[c] = df_analisis[c].fillna(df_analisis[c].max() + 1)

# El resto de columnas de análisis: un NaN significa "sin actividad histórica" -> se rellena con 0
col_valor = [c for c in df_analisis.columns if c not in ['Asesor'] + col_raking]
df_analisis[col_valor] = df_analisis[col_valor].fillna(0)

df_analisis.isnull().sum()

Asesor                   0
LeadsTotal               0
LeadsTotalAjustado       0
RakingLeads              0
RakingLeadsAjustado      0
LeadsPromXMes            0
TVentasPiso              0
Factor                   0
TotalVentasHistoricas    0
RakingVentas             0
DependenciaPiso          0
HistoricoPDM             0
HistoricoSDC             0
ConversionPDM            0
ConversionSDC            0
FactorPDM                0
FactorSDC                0
RakingPDM                0
RakingSDC                0
dtype: int64

In [6]:
# --- Verificación final ---
print("Nulos en df_registro:", df_registro.isnull().sum().sum())
print("Nulos en df_analisis:", df_analisis.isnull().sum().sum())

Nulos en df_registro: 0
Nulos en df_analisis: 0


In [7]:
# --- Exportar a CSV ---
df_registro.to_csv('basedeleads_registroleads.csv', index=False)
df_analisis.to_csv('basedeleads_analisisdesempeno.csv', index=False)